# Paper vs Backtest — Position Comparison (Phase 2.6 Tool 1 scaffold)

Loads paper-trade positions from PostgreSQL for a given run window,
re-runs the same strategy + config in `BacktestEngine` over the same
window's data, then nearest-matches paper × backtest positions by
entry time and reports entry-time / entry-price / PnL gaps.

Answers the Phase 2.6 keystone question: *how much do our backtests
lie about what live trading will actually look like?*

**Modes:**
- `USE_SYNTHETIC_DATA = True` (default) — runs end-to-end on a synthetic
  bar series with simulated paper-position lag/slippage injected. Useful
  for scaffold testing before Stage B real data exists.
- `USE_SYNTHETIC_DATA = False` — loads positions for `RUN_ID` from PG
  and re-runs `MACross` over the same bars. The `RUN_ID` and `INSTRUMENT_ID`
  must be set in cell 2; this notebook is standalone (does not chain off
  `review_live_run.ipynb`).


In [ ]:
# Cell 1: Imports + config

from datetime import UTC, datetime
from decimal import Decimal

import asyncpg
import matplotlib.pyplot as plt
import pandas as pd

from src.config.settings import get_settings
from src.core import bar_type_str, get_venue_config
from utils import (
    compare_position_streams,
    load_backtest_data,
    load_paper_positions,
    run_backtest_position_stream,
)

settings = get_settings()
CATALOG_PATH = '../data/catalog'
print(f'DB: {settings.postgres_host}:{settings.postgres_port}/{settings.postgres_db}')


In [ ]:
# Cell 2: Mode toggle + run identification

USE_SYNTHETIC_DATA = True   # flip to False once Stage B run_id exists

# REAL mode: set these from a row in strategy_runs.
# (Use review_live_run.ipynb cell 2 to list recent runs and pick one.)
RUN_ID         = ''                                  # UUID string
INSTRUMENT_ID  = 'BTC-USD-PERP.HYPERLIQUID'
BAR_INTERVAL   = '4h'

# Strategy hyperparameters — MUST match what paper actually ran with.
P26_FAST = 10
P26_SLOW = 40
P26_MA_TYPE = 'EMA'

# Match tolerance for entry-time alignment.
MATCH_TOL_BARS = 1.0
BAR_SECONDS = {'1m': 60, '5m': 300, '15m': 900, '1h': 3600, '4h': 14400, '1d': 86400}[BAR_INTERVAL]

print(f'mode: {"SYNTHETIC" if USE_SYNTHETIC_DATA else "REAL"}')
print(f'params: fast={P26_FAST} slow={P26_SLOW} ma_type={P26_MA_TYPE}')
print(f'match tol: {MATCH_TOL_BARS} bars ({MATCH_TOL_BARS * BAR_SECONDS}s)')


In [ ]:
# Cell 3: Build paper + backtest position streams

if USE_SYNTHETIC_DATA:
    # Synthetic mode — run two backtests over the same 120-bar BTC perp
    # series, then mutate the 'paper' stream to inject a 1-bar entry lag
    # AND +20 bps adverse slippage on every entry. Demonstrates the
    # canonical Phase 2.6 finding shape end-to-end.
    from nautilus_trader.model.data import Bar, BarType
    from nautilus_trader.model.objects import Price, Quantity
    from nautilus_trader.test_kit.providers import TestInstrumentProvider

    _inst = TestInstrumentProvider.btcusdt_perp_binance()
    _bt_str = bar_type_str(str(_inst.id), '1d')
    _bar_type = BarType.from_str(_bt_str)
    _closes = (
        [100.0 - i for i in range(30)]
        + [70.0 + i for i in range(60)]
        + [130.0 - i for i in range(30)]
    )
    _base = int(datetime(2024, 1, 1, tzinfo=UTC).timestamp() * 1e9)
    _one_day = 86_400 * 10**9
    _px_fmt = f'{{:.{int(_inst.price_precision)}f}}'
    _qty_fmt = f'{{:.{int(_inst.size_precision)}f}}'
    _bars = [
        Bar(_bar_type,
            Price.from_str(_px_fmt.format(c-0.5)),
            Price.from_str(_px_fmt.format(c+1.0)),
            Price.from_str(_px_fmt.format(c-1.0)),
            Price.from_str(_px_fmt.format(c)),
            Quantity.from_str(_qty_fmt.format(1.0)),
            _base + i * _one_day, _base + i * _one_day)
        for i, c in enumerate(_closes)
    ]
    _vc = get_venue_config('HYPERLIQUID_PERP')
    backtest_positions = run_backtest_position_stream(
        instrument=_inst, bars=_bars, venue_config=_vc,
        fast_period=3, slow_period=5, ma_type='EMA', leverage=1,
    )
    # Inject 1-bar entry lag + 20bps adverse slippage on opens.
    paper_positions = backtest_positions.copy()
    paper_positions['ts_opened'] = paper_positions['ts_opened'] + pd.Timedelta(days=1)
    paper_positions['ts_closed'] = paper_positions['ts_closed'] + pd.Timedelta(days=1)
    paper_positions['avg_px_open'] = paper_positions['avg_px_open'].apply(
        lambda d: d * Decimal('1.0020')
    )
    # Recompute realized_pnl crudely: paper PnL = bt PnL - entry slippage cost
    # (px_open went up 20bps; long PnL drops by 20bps × notional).
    paper_positions['realized_pnl'] = paper_positions.apply(
        lambda r: r['realized_pnl'] - Decimal('0.0020') * r['avg_px_open'] * r['peak_qty'],
        axis=1,
    )
    BAR_SECONDS = 86_400  # synthetic series uses 1d bars
else:
    # REAL mode — load paper positions from PG, re-run backtest over the
    # same window using catalog data.
    if not RUN_ID:
        raise RuntimeError('Set RUN_ID + INSTRUMENT_ID in cell 2 first.')
    paper_positions = await load_paper_positions(settings.postgres_dsn, RUN_ID)
    if paper_positions.empty:
        raise RuntimeError(f'No positions in PG for run {RUN_ID}.')
    win_start = paper_positions['ts_opened'].min().isoformat()
    win_end = paper_positions['ts_closed'].max().isoformat()
    bar_type_string = bar_type_str(INSTRUMENT_ID, BAR_INTERVAL)
    _vc = get_venue_config('HYPERLIQUID_PERP')
    instrument, bars = load_backtest_data(
        CATALOG_PATH, INSTRUMENT_ID, bar_type_string,
        venue_config=_vc, date_start=win_start, date_end=win_end,
    )
    backtest_positions = run_backtest_position_stream(
        instrument=instrument, bars=bars, venue_config=_vc,
        fast_period=P26_FAST, slow_period=P26_SLOW, ma_type=P26_MA_TYPE,
    )

print(f'paper positions:    {len(paper_positions):>6}')
print(f'backtest positions: {len(backtest_positions):>6}')


In [ ]:
# Cell 4: Match + headline gap summary

matched = compare_position_streams(
    paper_positions, backtest_positions,
    tol=MATCH_TOL_BARS, tol_unit='bars', bar_seconds=BAR_SECONDS,
)

n_matched = len(matched)
n_paper_unmatched = matched.attrs['paper_unmatched']
n_bt_unmatched = matched.attrs['bt_unmatched']

print('Phase 2.6 Tool 1 — paper vs backtest position comparison')
print('=' * 60)
print(f'Matched pairs:        {n_matched}')
print(f'Paper-only positions: {n_paper_unmatched}  (positions paper took but backtest did not)')
print(f'Backtest-only:        {n_bt_unmatched}  (positions backtest took but paper did not)')

if not matched.empty:
    print()
    print('Gap distributions (paper - backtest):')
    print(f'  entry_time_gap_s :  median={matched["entry_time_gap_s"].median():>10.1f}s  '
          f'mean={matched["entry_time_gap_s"].mean():>10.1f}s')
    print(f'  entry_px_gap_bps :  median={matched["entry_px_gap_bps"].median():>10.3f}   '
          f'mean={matched["entry_px_gap_bps"].mean():>10.3f}')
    print(f'  pnl_gap (USDC)   :  median={matched["pnl_gap"].median():>10.3f}    '
          f'sum={matched["pnl_gap"].sum():>10.3f}')

matched.head(10)


In [ ]:
# Cell 5: Gap histograms

if matched.empty:
    print('No matched pairs — nothing to plot.')
else:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    ax1, ax2, ax3 = axes

    ax1.hist(matched['entry_time_gap_s'] / 60, bins=30, color='#4C9AFF', edgecolor='white')
    ax1.axvline(0, color='red', linestyle='--', linewidth=1)
    ax1.set_title(f'Entry-time gap (paper - bt), minutes  |  median={matched["entry_time_gap_s"].median()/60:.1f}')
    ax1.set_xlabel('minutes')

    ax2.hist(matched['entry_px_gap_bps'], bins=30, color='#FF8B00', edgecolor='white')
    ax2.axvline(0, color='red', linestyle='--', linewidth=1)
    ax2.set_title(f'Entry-price gap, bps  |  median={matched["entry_px_gap_bps"].median():.2f}')
    ax2.set_xlabel('bps')

    ax3.hist(matched['pnl_gap'], bins=30, color='#36B37E', edgecolor='white')
    ax3.axvline(0, color='red', linestyle='--', linewidth=1)
    ax3.set_title(f'Realized PnL gap (USDC)  |  total={matched["pnl_gap"].sum():.2f}')
    ax3.set_xlabel('USDC')

    plt.tight_layout()
    plt.show()


In [ ]:
# Cell 6: Top-N worst entry slippage rows

if not matched.empty:
    top = matched.assign(abs_px_gap=matched['entry_px_gap_bps'].abs()) \
                 .sort_values('abs_px_gap', ascending=False) \
                 .head(10) \
                 .drop(columns=['abs_px_gap'])
    print('Top 10 matched positions by |entry_px_gap_bps|:')
    display(top)
else:
    print('No matched pairs.')

# Total realized PnL on each side, plus the absolute haircut.
if not matched.empty:
    paper_total = matched['paper_pnl'].astype(float).sum()
    bt_total = matched['bt_pnl'].astype(float).sum()
    haircut_pct = (
        (paper_total - bt_total) / abs(bt_total) * 100 if bt_total else 0.0
    )
    print()
    print(f'Paper realized PnL (matched):    {paper_total:>10.2f} USDC')
    print(f'Backtest realized PnL (matched): {bt_total:>10.2f} USDC')
    print(f'Haircut: {haircut_pct:+.1f}% (negative = paper underperformed backtest)')


In [ ]:
# Cell 7: Verdict snapshot for paste into PHASE_2_6_VERDICT.md (when it exists)

if matched.empty:
    print('No matched pairs — no verdict to write.')
else:
    paper_total = matched['paper_pnl'].astype(float).sum()
    bt_total = matched['bt_pnl'].astype(float).sum()
    haircut_pct = (
        (paper_total - bt_total) / abs(bt_total) * 100 if bt_total else 0.0
    )
    verdict = f"""## Backtest accuracy — measured (Tool 1)

| Metric | Value |
|---|---|
| Matched position pairs | {n_matched} |
| Paper-only positions (signal-skip) | {n_paper_unmatched} |
| Backtest-only positions | {n_bt_unmatched} |
| Entry-time gap median | {matched['entry_time_gap_s'].median():.0f}s |
| Entry-time gap p95 | {matched['entry_time_gap_s'].abs().quantile(0.95):.0f}s |
| Entry-price gap median | {matched['entry_px_gap_bps'].median():.2f} bps |
| Entry-price gap p95 | {matched['entry_px_gap_bps'].abs().quantile(0.95):.2f} bps |
| PnL gap total | {matched['pnl_gap'].sum():.2f} USDC |
| Realized PnL (paper / backtest) | {paper_total:.2f} / {bt_total:.2f} USDC |
| **Haircut** | **{haircut_pct:+.1f}%** |
"""
    print(verdict)
